# S0 · Mirror SynthRAD2025 from Zenodo to Hugging Face (run once)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/sct-range/blob/main/notebooks/s0_zenodo_to_huggingface.ipynb)

Downloading the 27 GB training set from Zenodo takes hours outside a data centre. This notebook copies it, once, to
a **public Hugging Face dataset**; every other notebook and `sctrange-download` then read the data from there only
(`sctrange.download`, parallel and resumable downloads, pinned tag).

For each archive of Zenodo record [15373853](https://doi.org/10.5281/zenodo.15373853) (training set v2,
`synthRAD2025_Task1_Train.zip` MR → CT and `synthRAD2025_Task2_Train.zip` CBCT → CT):

1. stream it from Zenodo with HTTP range resumption and check the MD5 published by Zenodo;
2. extract it to the layout the code expects, `Task<k>/<region>/<case_id>/{ct,mask,mr|cbct}.mha`, then delete the
   zip;
3. check every case (files present, identical geometry, isotropic in-plane spacing, non-empty body mask, CT within
   [-1024, 3071] HU) into `manifest.csv`;
4. upload the task with `upload_large_folder` (parallel, resumable), compare the size of every published file with
   the local one, then publish `manifest.csv` and `Task<k>/zenodo.json` (archive name, MD5, counts), which marks the
   task as complete;
5. free the disk before the next archive (peak use: one archive and its extracted cases, about 30 GB).

When both tasks are complete, the dataset card is written and the tag read by `sctrange` (`v2`) is set on the final
commit; the last section reads one case of each task back from Hugging Face.

**Rerunning is safe.** Completed tasks are skipped (their `zenodo.json` is on Hugging Face), an interrupted download
resumes, an extracted archive is not downloaded again and `upload_large_folder` only sends missing files.

**Before running** (Colab, a CPU runtime is enough):

- create a Hugging Face token with *write* access (huggingface.co → Settings → Access Tokens) and store it as the
  Colab secret `HF_TOKEN` (key icon in the left bar, notebook access on); without it the token is asked for;
- check `HF_REPO_ID` below: its namespace must be your Hugging Face user or one of your organisations. `sctrange`
  reads `{HF_DATASET}` by default (`sctrange.download.HF_DATASET`); for another repository, set `SCT_HF_DATASET`
  or change that constant;
- *Runtime → Run all*. Expect one to two hours, mostly transfers.

**Licence.** The data are CC BY-NC 4.0 (non-commercial use, attribution). The mirror keeps the licence, the source
and the citation in its dataset card.

Once the run ends with every check passed, delete this notebook from the repository
(`git rm notebooks/s0_zenodo_to_huggingface.ipynb`); nothing else depends on it.

### Setup · install `sctrange` from the repository

Run this cell first. Inside a clone of [hashirama21/sct-range](https://github.com/hashirama21/sct-range) it installs that
clone in editable mode (`pip install -e ".[dose]"`), so the notebook always runs the repository's code, local edits
included. Opened on its own, from the Colab or Kaggle badge or on another machine, it first downloads the repository
from GitHub (no git needed) to `/content/sct-range` on Colab, `/kaggle/tmp/sct-range` on Kaggle (so that
`/kaggle/working` only receives outputs) or `./sct-range` elsewhere, then installs it. On Kaggle, enable
*Settings → Internet* first. `SCTRANGE_REF` selects a branch, tag or commit (default `main`), `SCTRANGE_HOME` the
destination; pip only runs when `sctrange` is missing or installed from elsewhere (`SCTRANGE_REINSTALL=1` forces it).

In [ ]:
import importlib
import importlib.util
import io
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/hashirama21/sct-range"
REPO_REF = os.environ.get("SCTRANGE_REF", "main")
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir()


def find_repo(start: Path) -> Path | None:
    return next((p for p in (start, *start.parents) if (p / "pyproject.toml").exists() and (p / "src" / "sctrange").is_dir()),
                None)


def download_repo(destination: Path) -> Path:
    """Fetch ``REPO_REF`` of the GitHub repository as a zip archive and unpack it to ``destination``."""
    for attempt in range(4):
        try:
            with urllib.request.urlopen(f"{REPO_URL}/archive/{REPO_REF}.zip", timeout=120) as response:
                archive = zipfile.ZipFile(io.BytesIO(response.read()))
            break
        except OSError as error:
            if attempt == 3:
                raise
            print(f"download failed ({error}); retrying")
            time.sleep(5 * (attempt + 1))
    top = archive.namelist()[0].split("/")[0]
    staging = destination.with_name(destination.name + ".partial")
    shutil.rmtree(staging, ignore_errors=True)
    archive.extractall(staging)
    destination.parent.mkdir(parents=True, exist_ok=True)
    (staging / top).rename(destination)
    shutil.rmtree(staging, ignore_errors=True)
    return destination


REPO_ROOT = find_repo(Path.cwd())
if REPO_ROOT is None:
    default_home = (Path("/content/sct-range") if ON_COLAB else Path("/kaggle/tmp/sct-range") if ON_KAGGLE
                    else Path.cwd() / "sct-range")
    REPO_ROOT = Path(os.environ.get("SCTRANGE_HOME", default_home)).resolve()
    if find_repo(REPO_ROOT) is None:
        print(f"downloading {REPO_URL} @ {REPO_REF} to {REPO_ROOT}")
        download_repo(REPO_ROOT)
    os.chdir(REPO_ROOT / "notebooks")
SOURCE_DIR = (REPO_ROOT / "src").resolve()


def installed_from_repo() -> bool:
    spec = importlib.util.find_spec("sctrange")
    return spec is not None and spec.origin is not None and Path(spec.origin).resolve().is_relative_to(SOURCE_DIR)


if os.environ.get("SCTRANGE_REINSTALL") == "1" or not installed_from_repo():
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--editable", f"{REPO_ROOT}[dose]"]
    for attempt in range(1, 4):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0:
            break
        print(f"pip install failed (attempt {attempt}/3):\n{result.stderr[-3000:]}")
        time.sleep(10 * attempt)
    else:
        raise RuntimeError("pip could not install sctrange; see the messages above (network? Kaggle: enable Internet)")
# an editable install is only picked up by new processes; make it visible to this kernel too
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
for module in [name for name in sys.modules if name == "sctrange" or name.startswith("sctrange.")]:
    del sys.modules[module]
importlib.invalidate_caches()

import sctrange

assert Path(sctrange.__file__).resolve().is_relative_to(SOURCE_DIR), f"sctrange imported from {sctrange.__file__}"
print(f"sctrange {sctrange.__version__} installed from {REPO_ROOT.name}/src (editable)")

### Parameters

Set them here and run all cells (on Colab this cell is a form).

In [ ]:
# @title Parameters { display-mode: "form" }
HF_REPO_ID = "developper669/synthrad2025"  # @param {type:"string"}  public dataset created or completed here
HF_TAG = "v2"  # @param {type:"string"}  tag read by sctrange (sctrange.download.HF_REVISION)
TASKS = "1,2"  # @param ["1,2", "1", "2"]
WORK_DIR = "/content/zenodo"  # @param {type:"string"}  scratch disk for one archive and its cases (~30 GB)
UPLOAD_WORKERS = 16  # @param {type:"integer"}

In [ ]:
import hashlib
import json
import shutil
import urllib.error
import urllib.request
from dataclasses import dataclass
from pathlib import PurePosixPath

import pandas as pd
from huggingface_hub import HfApi
from huggingface_hub.errors import EntryNotFoundError
from IPython.display import display

from sctrange.data import CASE_ID_PATTERN, discover_cases
from sctrange.download import HF_DATASET, HF_REVISION, TASK_DIRS, check_case, read_manifest, validate_dataset

ZENODO_API = "https://zenodo.org/api/records/{record}"
ZENODO_RECORD = "15373853"
TASKS = tuple(sorted(set(TASKS.split(","))))
WORK = Path(WORK_DIR)
ARCHIVES, DATASET, REMOTE = WORK / "archives", WORK / "dataset", WORK / "remote"
for folder in (ARCHIVES, DATASET, REMOTE):
    folder.mkdir(parents=True, exist_ok=True)


def hf_token() -> str:
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    if ON_COLAB:
        try:
            from google.colab import userdata  # type: ignore[import-not-found]

            if token := userdata.get("HF_TOKEN"):
                return token
        except Exception as error:  # secret missing or notebook access not granted
            print(f"Colab secret HF_TOKEN not available ({type(error).__name__})")
    from getpass import getpass

    return getpass("Hugging Face token with write access: ")


api = HfApi(token=hf_token())
me = api.whoami()
namespaces = {me["name"], *(org["name"] for org in me.get("orgs", []))}
if HF_REPO_ID.split("/")[0] not in namespaces:
    raise PermissionError(f"{HF_REPO_ID}: the token belongs to {me['name']} (organisations: "
                          f"{', '.join(sorted(namespaces - {me['name']})) or 'none'}); change HF_REPO_ID")
if (HF_REPO_ID, HF_TAG) != (HF_DATASET, HF_REVISION):
    print(f"note: sctrange reads {HF_DATASET}@{HF_REVISION} by default; set SCT_HF_DATASET={HF_REPO_ID} and "
          f"SCT_HF_REVISION={HF_TAG} (or change sctrange.download) so that the notebooks use this mirror")
print(f"Hugging Face user {me['name']} · target {HF_REPO_ID}@{HF_TAG} · tasks {', '.join(TASKS)} · "
      f"{shutil.disk_usage(WORK).free / 2**30:.0f} GiB free in {WORK}")

### Zenodo: file list, resumable download, extraction

Only this notebook talks to Zenodo. Archive members are mapped to the canonical layout whatever their nesting in the
zip (`synthRAD2025_Task1_Train/Task1/AB/1ABA005/ct.mha` → `Task1/AB/1ABA005/ct.mha`; per-region overview folders keep
their path from `Task<k>` on); members that would escape the target directory are refused.

In [ ]:
CHUNK = 8 << 20


@dataclass(frozen=True)
class RemoteFile:
    key: str
    url: str
    size: int
    md5: str | None

    @property
    def task(self) -> str | None:
        return next((task for task, name in TASK_DIRS.items() if name.lower() in self.key.lower()), None)


def _open(url: str, headers: dict[str, str] | None = None, timeout: float = 60.0):
    return urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "sctrange", **(headers or {})}),
                                  timeout=timeout)


def list_record_files(record: str = ZENODO_RECORD, retries: int = 6) -> list[RemoteFile]:
    """Files, sizes and MD5 of a public Zenodo record (REST API, retried on network errors)."""
    for attempt in range(retries + 1):
        try:
            with _open(ZENODO_API.format(record=record)) as response:
                payload = json.load(response)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError) as error:
            if isinstance(error, urllib.error.HTTPError) and error.code < 500 or attempt == retries:
                raise
            print(f"Zenodo API: {error}; retrying")
            time.sleep(min(2**attempt, 30))
    files = []
    for entry in payload.get("files", []):
        algorithm, _, digest = entry.get("checksum", "").partition(":")
        files.append(RemoteFile(entry["key"], entry["links"].get("content") or entry["links"]["self"],
                                int(entry["size"]), digest if algorithm == "md5" and digest else None))
    return files


def md5sum(path: Path) -> str:
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        while block := handle.read(CHUNK):
            digest.update(block)
    return digest.hexdigest()


def download_file(remote: RemoteFile, destination: Path, retries: int = 12) -> Path:
    """Stream ``remote`` to ``destination`` through a ``.part`` file resumed with HTTP ranges; check size and MD5."""
    if destination.exists() and destination.stat().st_size == remote.size:
        if remote.md5 is None or md5sum(destination) == remote.md5:
            return destination
        destination.unlink()
    partial = destination.with_name(destination.name + ".part")
    start, last = time.monotonic(), 0.0
    for attempt in range(retries + 1):
        offset = partial.stat().st_size if partial.exists() else 0
        if offset > remote.size:
            partial.unlink()
            offset = 0
        if offset == remote.size:
            break
        try:
            with _open(remote.url, {"Range": f"bytes={offset}-"} if offset else {}, timeout=120.0) as response:
                if offset and response.status != 206:
                    offset = 0  # the server ignored the range: start again
                with open(partial, "ab" if offset else "wb") as handle:
                    done = offset
                    while block := response.read(CHUNK):
                        handle.write(block)
                        done += len(block)
                        if time.monotonic() - last > 10 or done == remote.size:
                            last = time.monotonic()
                            rate = (done - offset) / max(last - start, 1e-6) / 2**20
                            print(f"\r{remote.key}: {done / 2**30:5.2f} / {remote.size / 2**30:.2f} GiB "
                                  f"({done / remote.size:6.1%}, {rate:5.1f} MiB/s)", end="", flush=True)
            if partial.stat().st_size == remote.size:
                break
        except (urllib.error.URLError, TimeoutError, ConnectionError, OSError) as error:
            if attempt == retries:
                raise RuntimeError(f"download of {remote.key} failed after {retries + 1} attempts") from error
            print(f"\n{remote.key}: {error}; retrying")
            time.sleep(min(2**attempt, 60))
            start = time.monotonic()
    print()
    if remote.md5 is not None and (actual := md5sum(partial)) != remote.md5:
        partial.unlink()
        raise RuntimeError(f"{remote.key}: MD5 {actual} differs from Zenodo's {remote.md5}; partial file removed")
    partial.replace(destination)
    return destination


def member_destination(member: str, task: str) -> PurePosixPath | None:
    """Canonical relative path of a zip member, or ``None`` to skip it."""
    parts = PurePosixPath(member.replace("\\", "/")).parts
    if not parts or member.endswith("/") or any(p in ("..", "") for p in parts) or parts[0].startswith("/"):
        return None
    if parts[0].startswith("__MACOSX") or parts[-1].startswith("._"):
        return None
    for index, part in enumerate(parts[:-1]):
        if match := CASE_ID_PATTERN.match(part):
            fields = match.groupdict()
            return PurePosixPath(TASK_DIRS[fields["task"]], fields["region"], part, *parts[index + 1:])
    for index, part in enumerate(parts):
        if part in TASK_DIRS.values():
            return PurePosixPath(*parts[index:])
    return PurePosixPath(TASK_DIRS[task], *parts[1:] if len(parts) > 1 else parts)


def extract_archive(archive: Path, root: Path, task: str) -> int:
    """Extract into the canonical layout; files already present with the right size are skipped."""
    root = root.resolve()
    written = 0
    with zipfile.ZipFile(archive) as bundle:
        for info in bundle.infolist():
            relative = None if info.is_dir() else member_destination(info.filename, task)
            if relative is None:
                continue
            target = (root / relative).resolve()
            if root not in target.parents:
                raise ValueError(f"refusing to extract {info.filename!r} outside {root}")
            if target.exists() and target.stat().st_size == info.file_size:
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(target.name + ".tmp")
            with bundle.open(info) as source, open(temporary, "wb") as sink:
                shutil.copyfileobj(source, sink, CHUNK)
            temporary.replace(target)
            written += 1
    return written


ARCHIVE_FILES = [f for f in list_record_files() if f.key.endswith(".zip") and f.task in TASKS]
if sorted(f.task for f in ARCHIVE_FILES) != list(TASKS):
    raise RuntimeError(f"Zenodo record {ZENODO_RECORD}: expected one archive per task {TASKS}, "
                       f"found {[f.key for f in ARCHIVE_FILES]}")
display(pd.DataFrame([{"archive": f.key, "task": f.task, "GiB": round(f.size / 2**30, 2), "md5": f.md5}
                      for f in ARCHIVE_FILES]))

### Hugging Face: repository, published state, dataset card

The repository is created public if missing (made public if it exists as private). A task counts as published when
its `Task<k>/zenodo.json` exists with the MD5 of the current Zenodo archive.

In [ ]:
api.create_repo(HF_REPO_ID, repo_type="dataset", private=False, exist_ok=True)
if api.repo_info(HF_REPO_ID, repo_type="dataset").private:
    api.update_repo_settings(HF_REPO_ID, private=False, repo_type="dataset")


def published_json(path: str) -> dict | None:
    try:
        return json.loads(Path(api.hf_hub_download(HF_REPO_ID, path, repo_type="dataset", local_dir=REMOTE,
                                                   force_download=True)).read_text())
    except EntryNotFoundError:
        return None


def published_sizes(prefix: str) -> dict[str, int]:
    """Size of every published file under ``prefix``."""
    try:
        return {entry.path: entry.size for entry in api.list_repo_tree(HF_REPO_ID, prefix, recursive=True,
                                                                     repo_type="dataset")
                if getattr(entry, "size", None) is not None and not hasattr(entry, "tree_id")}
    except EntryNotFoundError:
        return {}


def restore_published_manifest() -> None:
    """Start from the published manifest, so that the rows of tasks uploaded in an earlier session are kept."""
    try:
        shutil.copy2(api.hf_hub_download(HF_REPO_ID, "manifest.csv", repo_type="dataset", local_dir=REMOTE,
                                         force_download=True), DATASET / "manifest.csv")
    except EntryNotFoundError:
        pass


def dataset_card(manifest: pd.DataFrame) -> str:
    counts = manifest.groupby(["task", "region"]).size().unstack(fill_value=0)
    table = "\n".join(["| Task | " + " | ".join(counts.columns) + " | Total |",
                       "|---|" + "---|" * (len(counts.columns) + 1)]
                      + [f"| {TASK_DIRS[t]} ({'MR' if t == '1' else 'CBCT'} → CT) | "
                         + " | ".join(str(v) for v in row) + f" | {sum(row)} |" for t, row in counts.iterrows()])
    return f"""---
license: cc-by-nc-4.0
pretty_name: SynthRAD2025 training set (v2)
tags:
- medical
- radiotherapy
- synthetic-ct
- mri
- cbct
- ct
task_categories:
- image-to-image
size_categories:
- 1K<n<10K
viewer: false
---

# SynthRAD2025 training set (v2), extracted

Unofficial mirror of the SynthRAD2025 Grand Challenge training data, Zenodo record
[15373853](https://doi.org/10.5281/zenodo.15373853) (v2: CT and CBCT clipped to [-1024, 3071] HU), extracted and
checked so that cases can be downloaded selectively. The files are the original `.mha` images, unchanged; each
archive was verified against the MD5 published by Zenodo and every uploaded file against its local size.

**Licence: CC BY-NC 4.0**, as the original record: non-commercial use only, with attribution. Cite the SynthRAD2025
dataset (Thummerer et al., 2025, doi:10.5281/zenodo.15373853); challenge site:
[synthrad2025.grand-challenge.org](https://synthrad2025.grand-challenge.org).

## Layout

```
Task1/<region>/<case_id>/{{ct,mask,mr}}.mha     MR → CT
Task2/<region>/<case_id>/{{ct,mask,cbct}}.mha   CBCT → CT
Task<k>/<region>/overview/...                 overview images and acquisition parameters of the release
Task<k>/zenodo.json                           source archive, MD5 and counts
manifest.csv                                  integrity check of every case (geometry, spacing, mask, HU range)
```

Regions: AB (abdomen), HN (head and neck), TH (thorax); `case_id` = task, region, centre letter, number.

{table}

{int(manifest["ok"].sum())}/{len(manifest)} cases pass the integrity check of `manifest.csv`.

## Download

```python
from huggingface_hub import snapshot_download

snapshot_download("{HF_REPO_ID}", repo_type="dataset", revision="{HF_TAG}", local_dir="synthrad2025",
                  allow_patterns=["manifest.csv", "Task1/HN/**"])   # e.g. task 1, head and neck only
```

With [sct-range](https://github.com/hashirama21/sct-range): `sctrange-download --root synthrad2025 --task 1`.
"""


state = {f.task: published_json(f"{TASK_DIRS[f.task]}/zenodo.json") for f in ARCHIVE_FILES}
display(pd.DataFrame([{"task": f.task, "published": bool(state[f.task] and state[f.task].get("md5") == f.md5)}
                      for f in ARCHIVE_FILES]))

## Mirror every task

One archive at a time: download and MD5, extraction, zip deleted, integrity check, upload, size check of every
published file, then `manifest.csv` and `zenodo.json` published and the local cases deleted.

In [ ]:
for archive in ARCHIVE_FILES:
    task, folder = archive.task, TASK_DIRS[archive.task]
    previous = state[task]
    if previous and previous.get("md5") == archive.md5:
        print(f"{folder}: already published ({previous['cases']} cases), skipped")
        continue
    extracted = WORK / f".extracted-{archive.md5 or archive.size}"
    if not (extracted.exists() and (DATASET / folder).is_dir()):
        partial = ARCHIVES / (archive.key + ".part")
        needed = 2.1 * archive.size - (partial.stat().st_size if partial.exists() else 0)  # zip + extracted cases
        if (free := shutil.disk_usage(WORK).free) < needed:
            print(f"warning: {free / 2**30:.0f} GiB free, about {needed / 2**30:.0f} GiB needed for {archive.key}")
        path = download_file(archive, ARCHIVES / archive.key)
        print(f"{archive.key}: MD5 verified; extracting")
        written = extract_archive(path, DATASET, task)
        extracted.write_text(archive.key)
        path.unlink()
        print(f"{folder}: {written} files extracted, archive deleted")

    restore_published_manifest()
    manifest = validate_dataset(DATASET, (task,))
    display(manifest.groupby(["region", "center"])["ok"].agg(cases="size", valid="sum"))
    if not manifest["ok"].all():
        print("cases with problems (published as they are, flagged in manifest.csv):")
        display(manifest.loc[~manifest["ok"], ["case_id", "problems"]])

    local = {p.relative_to(DATASET).as_posix(): p.stat().st_size for p in (DATASET / folder).rglob("*")
             if p.is_file() and not p.name.endswith((".tmp", ".part"))}
    print(f"{folder}: uploading {len(local)} files, {sum(local.values()) / 2**30:.1f} GiB")
    api.upload_large_folder(HF_REPO_ID, DATASET, repo_type="dataset", allow_patterns=[f"{folder}/**"],
                            num_workers=UPLOAD_WORKERS, print_report_every=120)
    published = published_sizes(folder)
    mismatched = sorted(k for k, size in local.items() if published.get(k) != size)
    if mismatched:
        raise RuntimeError(f"{folder}: {len(mismatched)} files missing or of another size on Hugging Face, e.g. "
                           f"{mismatched[:5]}; rerun this cell (the upload resumes)")
    print(f"{folder}: all {len(local)} files published with their exact size")

    api.upload_file(path_or_fileobj=DATASET / "manifest.csv", path_in_repo="manifest.csv", repo_id=HF_REPO_ID,
                    repo_type="dataset", commit_message=f"manifest.csv: {folder}")
    record = {"source": f"https://doi.org/10.5281/zenodo.{ZENODO_RECORD}", "archive": archive.key, "md5": archive.md5,
              "bytes": archive.size, "files": len(local), "cases": int(len(manifest)),
              "valid_cases": int(manifest["ok"].sum()), "uploaded": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    api.upload_file(path_or_fileobj=json.dumps(record, indent=2).encode(), path_in_repo=f"{folder}/zenodo.json",
                    repo_id=HF_REPO_ID, repo_type="dataset", commit_message=f"{folder}: complete ({archive.key})")
    state[task] = record
    shutil.rmtree(DATASET / folder)
    extracted.unlink()
    print(f"{folder}: complete, local copy deleted · {shutil.disk_usage(WORK).free / 2**30:.0f} GiB free")

## Dataset card and tag

The card is written from the published manifest. The tag `HF_TAG` is set (or moved) on the final commit only when
every task of the Zenodo record is published, so that `sctrange` never reads a partial mirror.

In [ ]:
restore_published_manifest()
published_manifest = read_manifest(DATASET / "manifest.csv")
api.upload_file(path_or_fileobj=dataset_card(published_manifest).encode(), path_in_repo="README.md",
                repo_id=HF_REPO_ID, repo_type="dataset", commit_message="Dataset card")

all_tasks = {f.task for f in list_record_files() if f.key.endswith(".zip") and f.task}
complete = {task for task in all_tasks if published_json(f"{TASK_DIRS[task]}/zenodo.json")}
if complete == all_tasks:
    if any(tag.name == HF_TAG for tag in api.list_repo_refs(HF_REPO_ID, repo_type="dataset").tags):
        api.delete_tag(HF_REPO_ID, tag=HF_TAG, repo_type="dataset")
    api.create_tag(HF_REPO_ID, tag=HF_TAG, repo_type="dataset",
                   tag_message=f"SynthRAD2025 training v2, Zenodo {ZENODO_RECORD}")
    print(f"tag {HF_TAG} set: https://huggingface.co/datasets/{HF_REPO_ID}/tree/{HF_TAG}")
else:
    print(f"tag not set: task(s) {', '.join(sorted(all_tasks - complete))} not published yet (rerun with TASKS)")
display(published_manifest.groupby(["task", "region"])["ok"].agg(cases="size", valid="sum"))

## Read back from Hugging Face

The first case of each task is downloaded at the tag, as `sctrange` does, and checked again.

In [ ]:
check_root = WORK / "readback"
shutil.rmtree(check_root, ignore_errors=True)
first = published_manifest.sort_values("case_id").groupby("task").head(1)
api.snapshot_download(HF_REPO_ID, repo_type="dataset", revision=HF_TAG if complete == all_tasks else None,
                      local_dir=check_root,
                      allow_patterns=[f"{TASK_DIRS[r.task]}/{r.region}/{r.case_id}/*" for r in first.itertuples()])
readback = pd.DataFrame([{"case_id": c.case_id, "ok": c.ok, "problems": c.problems}
                         for c in map(check_case, discover_cases(check_root))])
display(readback)
assert len(readback) == len(first) and readback["ok"].all(), "read-back check failed"
shutil.rmtree(check_root)
print(f"mirror ready: https://huggingface.co/datasets/{HF_REPO_ID} · every notebook now reads it")

## Done

Every notebook (`ensure_dataset`) and `sctrange-download` now fetch the data from this mirror only. This notebook is
not needed any more: delete it from the repository.

```bash
git rm notebooks/s0_zenodo_to_huggingface.ipynb && git commit -m "Remove the one-time Zenodo to Hugging Face notebook" && git push
```